# Binaryzacja

### Zadanie domowe - binaryzacja adaptacyjna w oknach z interpolacją.

Pokazana w ramach podstawowej części ćwiczenia binaryzacja adaptacyjna działa dobrze, ale jest dość złożona obliczeniowo (choć oczywiście należy mieć świadomość, że implementację można zoptymalizować i wyeliminować pewne powtarzające się obliczenia).
Zbliżone rozwiązanie można również realizować w nieco innym wariancie - w oknach.
Ogólna idea jest następująca: wejściowy obraz dzielimy na nienachodzące (rozłączne) okna - wygodnie jest założyć, że są one kwadratowe i o rozmiarze będącym potęgą liczby 2.
W każdym z okien obliczamy próg - niech to będzie średnia i stosujemy do binaryzacji lokalnej.
Jak nietrudno się domyślić efekt nie będzie dobry, gdyż na granicach okien wystąpią artefakty.
Aby je wyeliminować należy zastosować interpolację, co zostanie szczegółowo opisane poniżej.
Warto zaznaczyć, że podobny mechanizm interpolacji stosowany jest w poznanym wcześniej algorytmie CLAHE.
Zainteresowane osoby odsyłam do artykułu na [Wikipedii](https://en.wikipedia.org/wiki/Adaptive_histogram_equalization) oraz do artykułu o metodzie CLAHE - Zuiderveld, Karel. “Contrast Limited Adaptive Histograph Equalization.” Graphic Gems IV. San Diego: Academic Press Professional, 1994. 474–485.



Na początek zaimplementujemy wariant metody bez interpolacji:
1. Wczytaj obraz _rice.png_.
2. W dwóch pętlach `for`, dla okien o ustalonym wymiarze $W$ (potęga 2), oblicz średnią:
- pętle powinny mieć krok $W$,
- wynik (tj. średnie) należy zapisać w pomocniczej tablicy,
- przydatny operator to `//` - dzielenie całkowitoliczbowe (*floor division*).

3. W kolejnych dwóch pętlach `for` (tym razem o kroku 1) przeprowadź binaryzację z wyznaczonymi progami.
   Tu oczywiście należy się sprytnie odwołać do wyników z tablicy pomocniczej.
   Wyświetl wyniki - czy jest on poprawny?
   Podpowiedź - błędy lepiej widać dla mniejszego rozmiaru okna (np. 16 x 16).

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
import os
import urllib.request
import ssl

base_url = "https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/rice.png"


urllib.request.urlretrieve(base_url, "rice.png")

rice = cv2.imread("rice.png", cv2.IMREAD_GRAYSCALE)

W = 16

h, w = rice.shape

mean_blocks = np.zeros((h // W, w // W), dtype=np.uint8)

for y in range(0, h, W):
    for x in range(0, w, W):
        block = rice[y:y+W, x:x+W]
        mean_value = np.mean(block)

        mean_blocks[y // W, x // W] = mean_value

binary = np.zeros_like(rice)

for y in range(h):
    for x in range(w):
        block_y = y // W
        block_x = x // W

        threshold = mean_blocks[block_y, block_x]

        if rice[y, x] > threshold:
            binary[y, x] = 255
        else:
            binary[y, x] = 0

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(rice, cmap="gray")
plt.title("Obraz oryginalny")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(binary, cmap="gray")
plt.title(f"Binaryzacja lokalna, W = {W}")
plt.axis("off")

plt.show()



4. Rozwiązaniem problemu artefaktów na obrazie jest zastosowanie interpolacji.
   Próg binaryzacji dla danego okna wyliczany jest na podstawie progów z sąsiednich okien.
   ![Ilustracja koncepcji interpolacji](https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/clahe_tile_interpolation.png)

   Koncepcja została przedstawiona na powyższym rysunku.
   Możliwe są 3 przypadki:
   - piksel leży w rogach obrazu (kolor czerwony) - wtedy za próg przyjmuje się wartość średniej obliczonej dla danego okna,
   - piksel leży na krawędzi obrazu (kolor zielony) - wtedy za próg przyjmuje się wartość obliczoną na podstawie średnich z dwóch sąsiednich okien,
   - piksel leży w środku (kolor fioletowy) - wtedy próg jest obliczany na podstawie 4 sąsiednich okien.

   Uwaga. Proszę zwrócić uwagę, że sprawa jest dość złożona.
   Obraz dzielimy na okna (dla nich liczymy średnią) i następnie każde z okien "wirtualnie" na cztery sub-okna (linie przerywane).
   To ułatwia znalezienie środków okien (czarne kwadraty), które są wykorzystywane w interpolacji.

5. Implementujemy interpolację.
   Potrzebujemy do tego znać progi (jeden, dwa lub cztery), ale dla przejrzystości obliczeń lepiej zawsze przyjąć cztery oraz odległości od rozważnego piksela do środka sąsiednich okien (też w ogólnym przypadku 4):
   - całość sprowadza się do określania pozycji piksela,
   - na początek rozważmy przypadek czterech narożników (kolor czerwony na rysunku) - trzeba napisać `if`, który je wyznaczy,
   - warto sprawdzić, czy nie popełniliśmy błędu i np. tymczasowo do obrazu wynikowego w tym miejscu przypisać wartość 255. Efekt powinien być taki, że widoczne będą tylko narożniki.
   - drugi przypadek do brzegi (kolor zielony) - postępujemy podobnie jak przy narożnikach, przy czym osobno wydzielamy brzegi pionowe i poziome. Tu też warto sobie obrazek "pokolorować".
   - na koniec wyznaczamy piksele w środku.
   - analizując poprawność proszę zwrócić uwagę na to, żeby nie było przerw pomiędzy obszarami.
   - mając podział możemy dla każdego z obszarów wyliczyć cztery progi ($t11, t12, t21, t22$):
        - dla narożników wartość ta będzie identyczna i wynosi po prostu `t11 =t[jT][iT]`, gdzie `iT=i//W` oraz `jT=j//W`.
          Uwaga. Proszę używać indeksów tymczasowych $jT,iT$, gdyż będą potrzebne w dalszych obliczeniach.
        - dla brzegów pionowych występują dwie wartości: okno bieżące i sąsiednie.
          Wyznaczenie współrzędnej poziomej jest proste (jak dla narożników).
          Nad współrzędną pionową trzeba się chwilę zastanowić - aby nie rozważać wielu przypadków można od bieżącej współrzędnej odjąć połowę rozmiaru okna i dopiero później wykonać dzielenie przez rozmiar okna.
          W ten sposób otrzymujemy indeks okna o mniejszej współrzędnej.
          Indeks drugiego uzyskamy dodając 1.
          Proszę się zastanowić dlaczego to działa - najlepiej to sobie rozrysować.
        - dla brzegów poziomych należy postąpić analogicznie,
        - obliczenia dla obszaru wewnątrz powinny być już oczywiste.
   - kolejny krok to wyliczenie odległości pomiędzy rozważanym pikselem, a czterema środkami.
     Przykładowo dla osi X wygląda to następująco: `dX1 = i - W/2 - iT*W` oraz `dX2 = (iT+1)*W - i-W/2`.
     Dla osi Y analogicznie.
     Ponownie proszę się zastanowić dlaczego to jest poprawne - najlepiej to sobie narysować.
   - ostatni krok to interpolacja dwuliniowa.
     Wykonamy ją w trzech krokach:
     - interpolacja w osi X dla dwóch górnych okien - sprowadza się ona do średniej ważonej pomiędzy wartościami $t11$ i $t12$, przy czym wagi to odpowiednio $dX2/W$ i $dX1/W$.
       Ponownie na podstawie rysunku proszę to przemyśleć.
     - interpolacja w osi X dla dolnych okien jest analogiczna,
     - interpolacja w osi Y również jest analogiczna, z tym, że wejściem są dwa wyniki interpolacji w poziomie.

6. "Kropka nad i" to oczywiście binaryzacja z wyznaczonym poprzez interpolację progiem - proszę dobrać rozmiar okna.
7. Na koniec proszę porównać na wspólnym rysunku poznane metody binaryzacji:
- Otsu,
- lokalna na podstawie średniej,
- lokalna Sauvoli,
- lokalna w oknach bez interpolacji,
- lokalna w oknach z interpolacją.

Proszę pod porównaniem, w osobnej sekcji *markdown*, krótko skomentować uzyskane wyniki.

In [ ]:
from skimage.filters import threshold_sauvola


rice = cv2.imread("rice.png", cv2.IMREAD_GRAYSCALE)

h, w = rice.shape
W = 32


t = np.zeros((h // W, w // W), dtype=np.float32)

for j in range(0, h, W):
    for i in range(0, w, W):
        block = rice[j:j+W, i:i+W]
        t[j // W, i // W] = np.mean(block)


without_interp = np.zeros_like(rice)

for j in range(h):
    for i in range(w):
        threshold = t[j // W, i // W]

        if rice[j, i] > threshold:
            without_interp[j, i] = 255
        else:
            without_interp[j, i] = 0


with_interp = np.zeros_like(rice)

for j in range(h):
    for i in range(w):

        if ((i < W // 2 and j < W // 2) or
            (i >= w - W // 2 and j < W // 2) or
            (i < W // 2 and j >= h - W // 2) or
            (i >= w - W // 2 and j >= h - W // 2)):

            iT = min(i // W, t.shape[1] - 1)
            jT = min(j // W, t.shape[0] - 1)

            threshold = t[jT, iT]

        elif i < W // 2 or i >= w - W // 2:

            if i < W // 2:
                iT = 0
            else:
                iT = t.shape[1] - 1

            jT = (j - W // 2) // W
            jT = max(0, min(jT, t.shape[0] - 2))

            t11 = t[jT, iT]
            t21 = t[jT + 1, iT]

            dY1 = j - (jT * W + W // 2)
            dY2 = (jT + 1) * W + W // 2 - j

            threshold = (t11 * dY2 + t21 * dY1) / W

        elif j < W // 2 or j >= h - W // 2:

            if j < W // 2:
                jT = 0
            else:
                jT = t.shape[0] - 1

            iT = (i - W // 2) // W
            iT = max(0, min(iT, t.shape[1] - 2))

            t11 = t[jT, iT]
            t12 = t[jT, iT + 1]

            dX1 = i - (iT * W + W // 2)
            dX2 = (iT + 1) * W + W // 2 - i

            threshold = (t11 * dX2 + t12 * dX1) / W

        else:

            iT = (i - W // 2) // W
            jT = (j - W // 2) // W

            iT = max(0, min(iT, t.shape[1] - 2))
            jT = max(0, min(jT, t.shape[0] - 2))

            t11 = t[jT, iT]
            t12 = t[jT, iT + 1]
            t21 = t[jT + 1, iT]
            t22 = t[jT + 1, iT + 1]

            dX1 = i - (iT * W + W // 2)
            dX2 = (iT + 1) * W + W // 2 - i

            dY1 = j - (jT * W + W // 2)
            dY2 = (jT + 1) * W + W // 2 - j

            top = (t11 * dX2 + t12 * dX1) / W
            bottom = (t21 * dX2 + t22 * dX1) / W

            threshold = (top * dY2 + bottom * dY1) / W

        # Binaryzacja
        if rice[j, i] > threshold:
            with_interp[j, i] = 255
        else:
            with_interp[j, i] = 0

_, otsu = cv2.threshold(
    rice, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)


mean_local = cv2.adaptiveThreshold(
    rice,
    255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY,
    31,
    5
)


sauvola_threshold = threshold_sauvola(rice, window_size=31, k=0.2)
sauvola = np.where(rice > sauvola_threshold, 255, 0).astype(np.uint8)


fig, ax = plt.subplots(2, 3, figsize=(16, 10))

ax[0, 0].imshow(otsu, cmap="gray")
ax[0, 0].set_title("Otsu")

ax[0, 1].imshow(mean_local, cmap="gray")
ax[0, 1].set_title("Lokalna średnia")

ax[0, 2].imshow(sauvola, cmap="gray")
ax[0, 2].set_title("Sauvola")

ax[1, 0].imshow(without_interp, cmap="gray")
ax[1, 0].set_title(f"Bez interpolacji, W={W}")

ax[1, 1].imshow(with_interp, cmap="gray")
ax[1, 1].set_title(f"Z interpolacją, W={W}")

ax[1, 2].imshow(rice, cmap="gray")
ax[1, 2].set_title("Obraz oryginalny")

for a in ax.ravel():
    a.axis("off")

plt.tight_layout()
plt.show()

Metoda Otsu działa globalnie, dlatego słabiej radzi sobie z nierównomiernym oświetleniem obrazu. 
Lokalna binaryzacja oparta o średnią lepiej dopasowuje próg do otoczenia piksela, ale czasami może nadmiernie wzmacniać szum.
Metoda Sauvoli zwykle daje najlepszy kontrast i dobrze zachowuje obiekty przy zmiennym tle. 
Binaryzacja w oknach bez interpolacji powoduje wyraźne prostokątne artefakty na granicach okien. Widać, że każdy blok ma inny próg. 
Zastosowanie interpolacji usuwa większość tych artefaktów i daje znacznie bardziej płynny wynik. Obiekty nie są już pocięte granicami okien, a próg zmienia się stopniowo między sąsiednimi obszarami. 